# 01 - Format benchmark data (OP3, Tahoe, Novartis) into LPM-style long parquet

Mirrors `lpm_style/notebooks_work/01_LPM_style_formatting_data.ipynb`, but with two differences:

1. **Outer 75/25 split applied first.** Before formatting, each benchmark is split 75/25 at the compound level (random, `seed=42`, `ratio=0.25`). Only the **75% outer-train** portion is written to disk. The 25% outer-test portion is the held-out OP3-style benchmark test set and is **not** saved here — the user already has the original `.h5ad` files for that.
   - OP3:      split unit is `sm_name` (matches `splits.ipynb`).
   - Tahoe:    split unit is `pubchem_cid` (matches `process_tahoe_for_op3_based_evaluation.ipynb`).
   - Novartis: split unit is `pubchem_cid` (matches `process_novartis_for_op3_based_evaluation.ipynb`).
2. Output extends the existing LPM cache without modifying the 9 datasets already present:
   - `lpm_style/.plib_cache/raw_datasets/op3/{cell_type}.parquet` — one per OP3 cell type (`de_train`+`de_test` are concatenated first, then partitioned).
   - `lpm_style/.plib_cache/raw_datasets/tahoe/{source_file}.parquet` — one per source `.h5ad` (each source = a distinct cell line / batch).
   - `lpm_style/.plib_cache/raw_datasets/novartis/{source_file}.parquet` — one per source `.h5ad`.

Memory: Tahoe/Novartis source `.h5ad`s are loaded fully into memory; the concatenated view is used **only** to compute the global outer-train CID set, then each source `.h5ad` is filtered and written separately so the per-cell-line structure is preserved downstream.

Important: the outer 75/25 split is computed **globally** per benchmark (over all `sm_name`s for OP3 / all `pubchem_cid`s for Tahoe/Novartis), not per cell line. Partitioning by cell line happens only at write time, so a compound never lands in different splits across cell lines.

In [13]:
import os
import anndata as ad
import numpy as np
import pandas as pd
from tqdm import tqdm

In [14]:
LAYER = 'logFC'
PLIBDATA_ROOT = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets'
BASE_PATH = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated'

OP3_TRAIN_PATH = '../../data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad'
OP3_TEST_PATH  = '../../data/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad'
OP3_PUBCHEM    = '../../files/df_pubchem_op3.csv'

OUTER_RATIO = 0.25
OUTER_SEED  = 42

# OP3 obs does not have pert_dose_uM / pert_time_h. Neurips-2023 is single-condition;
# verify these against the dataset documentation if you care about exact dose / time downstream.
OP3_DEFAULT_DOSE_UM = 1.0
OP3_DEFAULT_TIME_H  = 24.0

## Helpers (same as `lpm_style/notebooks_work/01`)

In [15]:
def filter_samples(adata):
    adata = adata.copy()
    perturbagen_mask = ~adata.obs['pubchem_cid'].isna()
    dose_mask = ~(adata.obs['pert_dose_uM'] == 0)
    return adata[perturbagen_mask & dose_mask].copy()

In [16]:
def construct_df(adata):
    adata = adata.copy()

    dataset = adata.obs['dataset'].to_numpy()
    context = adata.obs['cell_type'].to_numpy()
    perturbations = adata.obs['pubchem_cid'].to_numpy()
    dose = adata.obs['pert_dose_uM'].to_numpy()
    time = adata.obs['pert_time_h'].to_numpy()

    values = adata.layers[LAYER]
    readout_names = adata.var.index.to_numpy()

    n_perts, n_readouts = values.shape

    df = pd.DataFrame({
        'dataset':      np.repeat(dataset, n_readouts),
        'context':      np.repeat(context, n_readouts),
        'perturbation': np.repeat(perturbations, n_readouts),
        'log_dose':     np.log10(np.repeat(dose, n_readouts)),
        'time':         np.repeat(time, n_readouts),
        'readout':      np.tile(readout_names, n_perts),
        'value':        values.ravel(),
    })
    return df

In [17]:
def filter_readout(df):
    df = df.copy()
    na_mask = df.isna().any(axis=1)
    if df[na_mask].shape[0] != 0:
        print(f'  Missing values: {df[na_mask].shape[0]}')
    return df[~na_mask].copy()

In [18]:
def run_construct_df(adata):
    adata = filter_samples(adata)
    df = construct_df(adata)
    df = filter_readout(df)
    return df

## OP3-specific obs preparation

OP3 anndata does not have `dataset`, `pubchem_cid`, `pert_dose_uM`, `pert_time_h` columns. We add them so the LPM-style helpers above work unchanged.

In [19]:
def prep_op3(adata, df_emb_op3):
    adata = adata.copy()
    cid_map = dict(zip(df_emb_op3['perturbagen'].astype(str), df_emb_op3['pubchem_cid'].astype(str)))
    adata.obs['dataset']      = 'op3'
    adata.obs['pubchem_cid']  = adata.obs['sm_name'].astype(str).map(cid_map)
    adata.obs['pubchem_cid']  = adata.obs['pubchem_cid'].replace({'nan': np.nan, 'None': np.nan})
    adata.obs['pert_dose_uM'] = OP3_DEFAULT_DOSE_UM
    adata.obs['pert_time_h']  = OP3_DEFAULT_TIME_H
    return adata

## Outer 75/25 split helper

Same logic as `splits.ipynb` cell 12 and `process_tahoe`/`process_novartis` cell 32:
```python
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * 0.25), replace=False)
```
Returns the 75% (outer-train) set; the 25% (outer-test) is discarded since the user already has the original `.h5ad` files.

In [20]:
def outer_train_compounds(compounds, ratio=OUTER_RATIO, seed=OUTER_SEED):
    """Return the 75% outer-train compound set, using the same RNG call shape as the
    existing splits.ipynb / process_tahoe / process_novartis notebooks (global seed +
    np.random.choice on the input array as-is)."""
    compounds = np.asarray(compounds)
    np.random.seed(seed)
    test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)
    outer_train = set(compounds.tolist()) - set(test_sample.tolist())
    print(f'  total compounds={len(compounds)}, outer_train(75%)={len(outer_train)}, outer_test(25%, held-out)={len(test_sample)}')
    return outer_train

## Format OP3

`splits.ipynb` concatenates `de_train.h5ad` + `de_test.h5ad` first and then splits the combined set 75/25 on `sm_name` (globally, not per cell type). We follow that exactly for the split, but **write one parquet per `cell_type`** so the on-disk layout is symmetric with Tahoe/Novartis (one parquet per cell line / batch).

Workflow:
1. Concat `de_train.h5ad` + `de_test.h5ad`.
2. 75/25 outer split on the **global** set of unique `sm_name`s.
3. Filter the concatenated anndata to the outer-train `sm_name`s and add the LPM-style obs (`prep_op3`).
4. Loop over `cell_type`, run the LPM helper chain on each slice, write `op3/{cell_type}.parquet`.

In [23]:
op3_full = ad.concat(
    [ad.read_h5ad(OP3_TRAIN_PATH), ad.read_h5ad(OP3_TEST_PATH)],
    uns_merge='same',
)
df_emb_op3 = pd.read_csv(OP3_PUBCHEM)
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

print('OP3 outer split (sm_name level, global across cell types):')
op3_outer_train_sm = outer_train_compounds(op3_full.obs['sm_name'].unique())

op3_outer_train_adata = op3_full[op3_full.obs['sm_name'].isin(op3_outer_train_sm)].copy()


OP3 outer split (sm_name level, global across cell types):
  total compounds=140, outer_train(75%)=105, outer_test(25%, held-out)=35


In [32]:
def _safe(name: str) -> str:
    return str(name).replace(' ', '_').replace('/', '_').replace('+', 'plus')


op3_outer_train_adata = prep_op3(op3_outer_train_adata, df_emb_op3)
os.makedirs(f'{PLIBDATA_ROOT}/op3', exist_ok=True)
cell_types = sorted(op3_outer_train_adata.obs['cell_type'].unique().tolist())
print(f'OP3 cell types ({len(cell_types)}): {cell_types}')
for cell in cell_types:
    a = op3_outer_train_adata[op3_outer_train_adata.obs['cell_type'] == cell].copy()
    if a.n_obs == 0:
        print(f'  [skip] {cell}: no rows')
        continue
    df = run_construct_df(a)
    out = f'{PLIBDATA_ROOT}/op3/{_safe(cell)}.parquet'
    df.to_parquet(out)
    print(f'  wrote {out}: {len(df):,} rows, {df["perturbation"].nunique()} CIDs')

OP3 cell types (4): ['B cells', 'Myeloid cells', 'NK cells', 'T cells']
  wrote /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/op3/B_cells.parquet: 558,285 rows, 105 CIDs
  wrote /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/op3/Myeloid_cells.parquet: 537,017 rows, 101 CIDs
  wrote /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/op3/NK_cells.parquet: 552,968 rows, 104 CIDs
  wrote /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/op3/T_cells.parquet: 558,285 rows, 105 CIDs


## Format Tahoe and Novartis

Outer split on `pubchem_cid` (mirrors `process_tahoe` / `process_novartis`). Same LPM-style helper chain for the actual formatting.

Workflow per dataset:
1. Load every source `.h5ad` into memory (one file per cell line / batch).
2. Concat all source anndatas, drop rows with NaN `pubchem_cid`, and collect the **global** set of unique CIDs (across all cell lines combined).
3. 75/25 outer split on that global set (`np.random.seed(42)`, `np.random.choice`).
4. Filter each source `.h5ad` separately to the outer-train CIDs and write `{filename_base}.parquet` per cell line, so the per-cell-line structure is preserved downstream.

In [38]:
dict_paths = {
    'tahoe':    f'{BASE_PATH}/tahoe/deg_data/group_rep/full/qc_false/filter_min_cells_50/results/',
    'novartis': f'{BASE_PATH}/novartis_batch_2500/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
}

In [39]:
for key, path in dict_paths.items():
    print(f'=== {key} ===')
    files = sorted(os.listdir(path))
    print(f'loading {len(files)} source files into memory...')
    data_list = [ad.read_h5ad(f'{path}{f}') for f in tqdm(files, desc=f'{key} load')]

    data_full = ad.concat(data_list)
    data_full = data_full[~data_full.obs['pubchem_cid'].isna()].copy()
    data_full.obs['pubchem_cid'] = data_full.obs['pubchem_cid'].astype(str)
    compounds = np.array(data_full.obs['pubchem_cid'].unique())
    print(f'{key} outer split (pubchem_cid level, NaN dropped first):')
    outer_train_cids = outer_train_compounds(compounds)
    del data_full

    os.makedirs(f'{PLIBDATA_ROOT}/{key}', exist_ok=True)
    for i, (f, adata) in enumerate(zip(files, data_list)):
        adata.obs['pubchem_cid'] = adata.obs['pubchem_cid'].astype(str)
        a = adata[adata.obs['pubchem_cid'].isin(outer_train_cids)].copy()
        if a.n_obs == 0:
            print(f'  ({i+1}/{len(files)}) [skip] {f}: no rows after outer-train filter')
            continue
        df = run_construct_df(a)
        out = f'{PLIBDATA_ROOT}/{key}/{f.split("_de")[0]}.parquet'
        df.to_parquet(out)
        print(f'  ({i+1}/{len(files)}) {out}: {len(df):,} rows, {df["perturbation"].nunique()} CIDs, {df["context"].nunique()} contexts')
    del data_list

=== tahoe ===
loading 48 source files into memory...


tahoe load: 100%|██████████| 48/48 [14:50<00:00, 18.55s/it]
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


tahoe outer split (pubchem_cid level, NaN dropped first):
  total compounds=378, outer_train(75%)=284, outer_test(25%, held-out)=94
  (1/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0023.parquet: 21,886,650 rows, 284 CIDs, 1 contexts
  (2/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0028.parquet: 14,347,255 rows, 284 CIDs, 1 contexts
  (3/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0069.parquet: 23,072,384 rows, 284 CIDs, 1 contexts
  (4/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0099.parquet: 18,798,464 rows, 284 CIDs, 1 contexts
  (5/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0131.parquet: 22,295,500 rows, 284 CIDs, 1 contexts
  (6/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0152.parquet: 21,168,625 rows, 284 CIDs, 1 contexts
  (7/48) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
novartis load: 100%|██████████| 1/1 [02:40<00:00, 160.68s/it]
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


novartis outer split (pubchem_cid level, NaN dropped first):
  total compounds=3819, outer_train(75%)=2865, outer_test(25%, held-out)=954


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


  Missing values: 29907492
  (1/1) /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/novartis/CVCL_0042.parquet: 142,130,148 rows, 2865 CIDs, 1 contexts


In [59]:
df = pd.read_parquet('/home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets/tahoe/CVCL_0023.parquet')

In [61]:
df['perturbation'].unique()

array(['11557040', '5330286', '67462786', '86705695', '60750', '9887053',
       '138911318', '25262965', '10288191', '148195', '25145656',
       '58282870', '11167602', '51039094', '42611257', '11707110',
       '10184653', '51003603', '2747117', '118598754', '446727',
       '50922675', '135397144', '58507717', '23582824', '16747388',
       '118959080', '16124208', '24788740', '70817911', '44582816',
       '5327711', '10138980', '121241171', '118704762', '46205871',
       '45256693', '45256689', '2359994', '146681181', '71621331',
       '129073603', '24963252', '129893299', '2662', '5978', '25183872',
       '67448836', '53340664', '91885558', '135151360', '11960271',
       '445639', '130956', '2187', '46208890', '118701008', '24775005',
       '3064778', '117947097', '16923', '86295191', '73442', '1923',
       '9562059', '6256', '275182', '444795', '9082', '72205932',
       '6419992', '23665037', '6918111', '3440', '16760658', '170014',
       '1054', '19090', '9875462', '33

In [62]:
train_t = ad.read_h5ad('../../data_tmp/tahoe/de_train.h5ad')

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [67]:
len(set(train_t.obs['pubchem_cid']).intersection(set(df['perturbation'].unique())))

284

In [68]:
len(set(train_t.obs['pubchem_cid']))

284